# Jour 3 · La conversation : orchestration, tours de parole, attente

**Formation Agents vocaux de 0 à Z · AI&ML Factory**

Hier, vous avez ouvert le capot : l'oreille (STT), le cerveau (LLM) et la voix (TTS). Aujourd'hui, on les fait **travailler ensemble** : le pipeline, les tours de parole et les interruptions, puis l'attente et le transfert. Chaque module a sa section, qui peut s'exécuter seule grâce à sa **cellule de reprise**.

| Section | Module | Statut |
|---|---|---|
| 0 | Configuration | disponible |
| 8 | L'orchestration et le pipeline | disponible |
| 9 | Tours de parole et interruptions | disponible |
| 10 | Attente, objections, transfert et accents | disponible |

**Fil rouge** : Awa, l'assistante commerciale vocale de Baobab Software (entreprise fictive).

**Niveaux d'exercices**
- **[Guidé]** : code à compléter aux endroits marqués `# TODO`, avec un indice.
- **[Avancé]** : consigne ouverte, pour aller plus loin.

## 0. Configuration

Exécutez cette cellule une fois au début de la journée. Les modules 8 et 9 fonctionnent **entièrement hors ligne** : les briques et les signaux sont simulés, sans clé API.

In [ ]:
# Installation (décommentez si nécessaire, par exemple sur Colab)
# !pip install -q numpy matplotlib

import os, re, time, json, asyncio
import numpy as np
import matplotlib.pyplot as plt

NAVY, TEAL, AMBER, GRAY, LIGHT = "#19233E", "#1E8C95", "#F2A93B", "#C9CFDB", "#9CCFD3"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

CLE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print("Configuration prête.", "Clé OpenAI détectée." if CLE_OPENAI else "Mode hors ligne (aucune clé API détectée).")

---
## 8. L'orchestration et le pipeline : faire travailler les briques ensemble

**Objectifs**
1. Suivre le **parcours complet d'un tour** de parole, de la fin de la voix du prospect au premier son d'Awa.
2. Comprendre ce que `asyncio` apporte : **attendre sans bloquer**, lancer des tâches **en parallèle**.
3. Construire un **mini pipeline** de briques reliées par des **files** (`asyncio.Queue`).
4. Voir ce qui tourne **en parallèle** et ce qui reste **en séquence**, et lire un **budget de latence** brique par brique.
5. **[Avancé]** Annuler une réponse en cours quand le prospect coupe la parole.
6. **Instrumenter** Awa pour mesurer chaque brique.

**Durée** : 35 minutes d'exercices, puis l'atelier (instrumenter Awa) et le quiz (10 minutes). Tout fonctionne **hors ligne**.

**Attention aux chiffres** : les durées des briques simulées sont **fictives** (repère indicatif). Elles servent à comprendre la mécanique ; vos vraies mesures viennent de l'atelier.

In [ ]:
# Cellule de reprise du Module 8
import re, time, json, asyncio
import numpy as np, matplotlib.pyplot as plt
try:
    NAVY
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")
RED = "#C0392B"

ECHELLE = 0.1      # 1 000 ms « simulées » = 100 ms réelles : le notebook reste rapide

async def dormir(ms):
    """Attente non bloquante, en millisecondes simulées."""
    await asyncio.sleep(ms * ECHELLE / 1000)

print("Module 8 prêt.")

### 8.1 Le parcours d'un tour de parole

Quand le prospect se tait, un tour d'Awa commence. Voici les maillons, **dans l'ordre**, et ce que chacun fait :

| Maillon | Ce qui se passe | Où on l'a vu |
|---|---|---|
| **Détection de fin de tour** | on décide que le prospect a fini de parler | module 5, module 9 |
| **STT** | le texte final est prêt | module 5 |
| **LLM** | le premier mot, puis la première phrase | module 6 |
| **TTS** | le premier morceau d'audio | module 7 |
| **Réseau et lecture** | l'audio arrive dans l'oreille du prospect | module 3 |

Le silence que le prospect entend est la **somme** de ces maillons. Pour la maîtriser, on se donne un **budget** : une durée cible par maillon. Les valeurs ci-dessous sont un **repère indicatif** pour le fil rouge (pas une norme) :

In [ ]:
BUDGET_MS = {                       # repère indicatif, à ajuster avec vos mesures
    "détection de fin de tour": 250,
    "STT final": 100,
    "LLM (première phrase)": 300,
    "TTS (premier son)": 200,
    "réseau et lecture": 150,
}
total = sum(BUDGET_MS.values())
for nom, ms in BUDGET_MS.items():
    print(f"{nom:28} {ms:5d} ms  {'#' * (ms // 25)}")
print(f"{'total visé':28} {total:5d} ms")

Lecture : on vise **environ une seconde** entre la fin de la voix du prospect et le début de la voix d'Awa. Le budget est une **hypothèse de départ** : vous le corrigerez avec vos mesures réelles en fin de module, puis au module 12 (évaluation).

### 8.2 `asyncio` en cinq minutes

Un agent vocal fait **beaucoup d'attente** : réseau, modèle, agenda. `asyncio` permet de **ne pas rester bloqué** pendant l'attente : tant qu'une tâche attend, une autre avance.

- Une **coroutine** se définit avec `async def` et s'attend avec `await`.
- `asyncio.gather(a(), b())` lance les deux **en même temps** et rend les résultats quand tout est fini.
- `asyncio.Queue` est une **file** : un maillon y dépose, le suivant y retire, sans que les deux se bloquent.

Dans un notebook, vous pouvez écrire `await ...` directement dans une cellule.

Exemple d'Awa : le prospect demande un créneau. Il faut interroger l'agenda (lent) **et** préparer une phrase d'attente (rapide). En séquence, les deux durées s'additionnent. En parallèle, on attend seulement la plus longue.

In [ ]:
async def agenda_lent():
    await dormir(400)
    return "créneaux libres"

async def preparer_phrase():
    await dormir(150)
    return "Je vérifie l'agenda."

async def chronometrer(coro):
    """Exécute une coroutine et rend (résultat, durée en ms simulées)."""
    t0 = time.perf_counter()
    resultat = await coro
    return resultat, (time.perf_counter() - t0) / ECHELLE * 1000

async def sequentiel():
    a = await agenda_lent()
    b = await preparer_phrase()
    return a, b

res, duree = await chronometrer(sequentiel())
print(f"En séquence : {duree:4.0f} ms  ->  {res}")

**[Guidé] Exercice 8.1** : écrivez `en_parallele()`, qui lance `agenda_lent()` et `preparer_phrase()` **en même temps**. Le résultat doit rester le même couple, mais la durée doit passer d'environ 550 ms à environ 400 ms.

*Indice* : `a, b = await asyncio.gather(coro1, coro2)` rend les résultats **dans l'ordre** où vous les avez donnés.

In [ ]:
async def en_parallele():
    # TODO : remplacer les deux lignes ci-dessous par un seul asyncio.gather
    a = await agenda_lent()
    b = await preparer_phrase()
    return a, b

res, duree = await chronometrer(en_parallele())
print(f"En parallèle : {duree:4.0f} ms  ->  {res}")
print("Gain obtenu." if duree < 470 else "Pas encore de gain : les deux tâches s'exécutent l'une après l'autre.")

Point à retenir : le parallélisme de `asyncio` est du **partage du temps d'attente**, pas du calcul sur plusieurs cœurs. Un calcul lourd écrit dans une coroutine **bloque tout le monde** : on le déporte dans un autre thread ou processus (`asyncio.to_thread`, vu dans `awa_agent.py` pour les appels HTTP).

### 8.3 Un mini pipeline avec des files

Maintenant, les quatre briques d'Awa en miniature, **reliées par des files** :

```
fin de parole -> [détection] -> [STT] -> q_texte -> [LLM] -> q_phrases -> [TTS] -> q_audio -> [lecture]
```

Chaque brique est une coroutine qui **lit** sa file d'entrée et **écrit** dans sa file de sortie. Un `None` dans une file veut dire « c'est fini ». Les durées sont **fictives** (repère indicatif) et suivent les ordres de grandeur des modules 5, 6 et 7.

In [ ]:
# Paramètres de la simulation (en ms simulées, valeurs fictives : repère indicatif)
PARAMS = {"detection": 250, "stt_final": 100, "llm_ttft": 300, "llm_ms_par_mot": 20,
          "tts_ttfb": 200, "tts_pas_frame": 50, "audio_ms_par_mot": 250, "frame_ms": 100, "reseau": 150}

REPONSE_AWA = ("Très bien, je note votre demande. "
               "Pour une quincaillerie de huit salariés, notre offre Essentiel à vingt-cinq mille francs CFA par mois suffit pour démarrer. "
               "Souhaitez-vous un rendez-vous mardi à neuf heures ?")
print(f"La réponse d'Awa fait {len(REPONSE_AWA.split())} mots, en 3 phrases.")

class Trace:
    """Note à quel moment (en ms simulées, depuis la fin de la parole du prospect) chaque événement a lieu."""
    def __init__(self):
        self.t0 = time.perf_counter()
        self.marques = {}
        self.intervalles = []          # (brique, début, fin)

    def maintenant(self):
        return (time.perf_counter() - self.t0) / ECHELLE * 1000

    def marquer(self, nom):
        self.marques.setdefault(nom, self.maintenant())

    def intervalle(self, brique, debut):
        self.intervalles.append((brique, debut, self.maintenant()))

**[Guidé] Exercice 8.2** : le LLM diffuse sa réponse **mot à mot**. Pour que le TTS démarre tôt, on coupe en **phrases** (idée du module 6). Écrivez `phrase_terminee(mot)`, qui renvoie `True` si le mot termine une phrase (il finit par `.`, `?` ou `!`).

*Indice* : `mot.endswith((".", "?", "!"))`. Avec la version de départ qui renvoie toujours `False`, la réponse part **d'un seul bloc** à la fin : ce sera votre point de comparaison.

In [ ]:
def phrase_terminee(mot):
    # TODO : True si le mot termine une phrase
    return False

print([phrase_terminee(m) for m in ["bonjour", "demande.", "heures", "?"]])

In [ ]:
# Les briques : chacune lit une file et écrit dans la suivante
async def brique_stt(q_texte, trace):
    d = trace.maintenant()
    await dormir(PARAMS["detection"])                     # le système attend pour décider que le prospect a fini
    trace.intervalle("détection", d); trace.marquer("fin de tour")
    d = trace.maintenant()
    await dormir(PARAMS["stt_final"])
    trace.intervalle("STT", d); trace.marquer("texte final")
    await q_texte.put("Je voudrais un rendez-vous mardi.")
    await q_texte.put(None)

async def brique_llm(q_texte, q_phrases, trace, coupe):
    texte = await q_texte.get()
    d = trace.maintenant()
    await dormir(PARAMS["llm_ttft"])
    trace.marquer("premier mot")
    courant, premiere_envoyee = [], False
    for mot in REPONSE_AWA.split():
        await dormir(PARAMS["llm_ms_par_mot"])
        courant.append(mot)
        if coupe(mot):
            await q_phrases.put(" ".join(courant)); courant = []
            if not premiere_envoyee:
                trace.marquer("première phrase"); trace.intervalle("LLM", d); premiere_envoyee = True
    if courant:                                            # reste du texte (ou tout le texte si on ne coupe jamais)
        await q_phrases.put(" ".join(courant))
        if not premiere_envoyee:
            trace.marquer("première phrase"); trace.intervalle("LLM", d)
    trace.marquer("fin du LLM")
    await q_phrases.put(None)

async def brique_tts(q_phrases, q_audio, trace):
    premiere = True
    while (phrase := await q_phrases.get()) is not None:
        d = trace.maintenant()
        await dormir(PARAMS["tts_ttfb"])                   # délai avant le premier son de cette phrase
        if premiere:
            trace.marquer("premier son"); trace.intervalle("TTS", d); premiere = False
        n_frames = max(1, round(len(phrase.split()) * PARAMS["audio_ms_par_mot"] / PARAMS["frame_ms"]))
        for i in range(n_frames):
            await q_audio.put((phrase, i))
            await dormir(PARAMS["tts_pas_frame"])          # le TTS produit plus vite que le temps réel
    await q_audio.put(None)

async def brique_lecture(q_audio, trace, joues):
    premier = True
    while (frame := await q_audio.get()) is not None:
        if premier:
            await dormir(PARAMS["reseau"])
            trace.marquer("audio entendu"); premier = False
        joues.append(frame)

async def lancer_pipeline(coupe, delai_max_ms=20000):
    """Fait tourner les quatre briques et rend la trace. Garde-fou : le pipeline ne doit pas tourner indéfiniment."""
    trace, joues = Trace(), []
    q_texte, q_phrases, q_audio = asyncio.Queue(), asyncio.Queue(), asyncio.Queue()
    briques = asyncio.gather(brique_stt(q_texte, trace), brique_llm(q_texte, q_phrases, trace, coupe),
                             brique_tts(q_phrases, q_audio, trace), brique_lecture(q_audio, trace, joues))
    try:
        await asyncio.wait_for(briques, timeout=delai_max_ms * ECHELLE / 1000)
    except asyncio.TimeoutError:
        print("Le pipeline ne se termine pas : une brique attend un None qui n'arrive jamais.")
    return trace, joues

# Comparaison : « séquentiel » (on ne coupe jamais, tout part d'un bloc) contre « pipeline » (on coupe en phrases)
trace_seq, _ = await lancer_pipeline(lambda mot: False)
trace_pip, joues_pip = await lancer_pipeline(phrase_terminee)
for nom, tr in (("séquentiel", trace_seq), ("pipeline", trace_pip)):
    print(f"{nom:11} : premier mot LLM {tr.marques['premier mot']:5.0f} ms   première phrase {tr.marques['première phrase']:5.0f} ms"
          f"   audio entendu {tr.marques['audio entendu']:5.0f} ms")

Lecture : dans la version « séquentielle » (qui est aussi ce que donne votre `phrase_terminee` de départ), le TTS attend **toute** la réponse du LLM. Dans le pipeline, il reçoit la **première phrase** dès qu'elle est prête, pendant que le LLM écrit la suite. Le gain est la durée de génération des phrases suivantes. Plus la réponse est longue, plus il est grand : c'est une raison de plus pour écrire court à l'oral (module 6).

### 8.4 Ce qui tourne en parallèle, ce qui reste en séquence

Le graphique suivant (un diagramme de Gantt) montre, pour le pipeline, **quand chaque brique travaille**. Lisez-le de gauche à droite : les barres qui se **recouvrent** sont des briques qui tournent en parallèle.

In [ ]:
def gantt(trace, titre, ax=None):
    couleurs = {"détection": GRAY, "STT": LIGHT, "LLM": TEAL, "TTS": AMBER}
    ax = ax or plt.gca()
    ordre = ["détection", "STT", "LLM", "TTS"]
    for brique, d, f in trace.intervalles:
        ax.barh(ordre.index(brique), f - d, left=d, color=couleurs[brique], edgecolor="white")
    ax.axvline(trace.marques["audio entendu"], color=RED, lw=1.5)
    ax.text(trace.marques["audio entendu"] + 15, 3.45, "audio entendu", color=RED, fontsize=9)
    ax.set_yticks(range(len(ordre))); ax.set_yticklabels(ordre); ax.invert_yaxis()
    ax.set_xlabel("ms après la fin de la parole du prospect (simulées)")
    ax.set_title(titre, color=NAVY)

fig, axes = plt.subplots(2, 1, figsize=(8, 5.2), sharex=True)
gantt(trace_seq, "Séquentiel : le TTS attend la fin du LLM", axes[0])
gantt(trace_pip, "Pipeline : le TTS démarre sur la première phrase", axes[1])
plt.tight_layout(); plt.show()

Ce qui est **en séquence**, par nature : la détection de fin de tour **avant** le STT final, le STT final **avant** le LLM, la première phrase du LLM **avant** le premier son. Ce qui est **en parallèle** : la génération de la suite pendant que le TTS parle, et tout ce qui n'est pas sur le chemin critique (journalisation, outils lents avec phrase d'attente).

LiveKit Agents va plus loin avec la **génération préemptive** : le LLM démarre sur le texte **provisoire** avant que la fin de tour soit confirmée, et on jette le résultat si le prospect continue de parler. C'est un pari : on gagne du temps quand on a raison, on dépense des appels LLM pour rien quand on a tort. Les options de `AgentSession` évoluent vite (dans la version 1.8.10, elles se regroupent dans `turn_handling`) : vérifiez-les dans la documentation le jour J.

### 8.5 Le budget de latence, brique par brique

On compare maintenant ce que le pipeline **mesure** au budget de 8.1. Chaque durée de brique se lit comme la **différence** entre deux marques.

**[Guidé] Exercice 8.3** : complétez `decomposer(marques)`, qui renvoie la durée de chaque maillon à partir des marques de la trace, puis `ecart_budget(mesure, budget)`, qui renvoie pour chaque maillon l'écart en ms (positif = dépassement).

*Indice* : « STT final » = `marques["texte final"] - marques["fin de tour"]`. La première phrase du LLM se compte depuis `texte final`.

In [ ]:
def decomposer(marques):
    return {
        "détection de fin de tour": marques["fin de tour"],
        "STT final": marques["texte final"] - marques["fin de tour"],
        "LLM (première phrase)": None,     # TODO : de « texte final » à « première phrase »
        "TTS (premier son)": None,         # TODO : de « première phrase » à « premier son »
        "réseau et lecture": None,         # TODO : de « premier son » à « audio entendu »
    }

def ecart_budget(mesure, budget):
    # TODO : pour chaque maillon, mesure - budget
    return {nom: 0.0 for nom in budget}

mesure = decomposer(trace_pip.marques)
ecarts = ecart_budget({k: v or 0 for k, v in mesure.items()}, BUDGET_MS)
for nom in BUDGET_MS:
    print(f"{nom:28} mesuré {mesure[nom] or 0:5.0f} ms   budget {BUDGET_MS[nom]:4d} ms   écart {ecarts[nom]:+5.0f} ms")

In [ ]:
def graphique_budget(mesure, budget):
    noms = list(budget)
    fig, ax = plt.subplots(figsize=(8, 3.6))
    gauche_m = gauche_b = 0
    couleurs = [GRAY, LIGHT, TEAL, AMBER, NAVY]
    for nom, c in zip(noms, couleurs):
        m = mesure.get(nom) or 0
        ax.barh(1, m, left=gauche_m, color=c, edgecolor="white", label=nom)
        ax.barh(0, budget[nom], left=gauche_b, color=c, edgecolor="white", alpha=0.55)
        if m > 60: ax.text(gauche_m + m / 2, 1, f"{m:.0f}", ha="center", va="center", color="white" if c in (TEAL, NAVY) else NAVY, fontsize=9)
        gauche_m += m; gauche_b += budget[nom]
    ax.set_yticks([0, 1]); ax.set_yticklabels(["budget", "mesuré"])
    ax.set_xlabel("ms entre la fin de la parole du prospect et l'audio entendu (simulées)")
    ax.legend(ncol=3, fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.28))
    plt.tight_layout(); plt.show()

graphique_budget(mesure, BUDGET_MS)

Ce graphique est l'outil de **décision** du module : il montre le maillon qui fait déraper le total. Ici, le LLM dépasse son budget parce que la réponse commence par une phrase de **six mots** après un TTFT de 300 ms. On a deux leviers : un modèle plus rapide, ou **commencer par une phrase plus courte** (par exemple « Très bien. »).

### 8.6 [Avancé] Annuler une réponse en cours

Le prospect coupe la parole en pleine réponse d'Awa (le *barge-in*, traité en détail au module 9). Côté code, il faut **arrêter net** : le TTS cesse de produire, la lecture s'arrête, les files se vident. En `asyncio`, on **annule les tâches** (`tache.cancel()`), ce qui lève `CancelledError` à l'endroit où chaque tâche attend.

**[Avancé] Exercice 8.4** : écrivez `repondre_avec_interruption(interrompre_apres_ms)`, qui lance le pipeline (avec `phrase_terminee`) en **tâches** (`asyncio.create_task`), attend `interrompre_apres_ms` ms simulées après le départ, puis **annule toutes les tâches** et attend qu'elles aient fini (`await asyncio.gather(*taches, return_exceptions=True)`). La fonction renvoie un dictionnaire :
- `"frames_joues"` : le nombre de frames lues au moment de l'annulation ;
- `"frames_apres"` : le nombre de frames lues **après** une attente supplémentaire de 300 ms (doit être **0** de plus) ;
- `"phrases_dites"` : les phrases dont **toutes** les frames ont été lues.

*Piège* : `CancelledError` ne doit jamais être avalé par un `except Exception` ; et les files peuvent contenir des frames qui ne seront jamais lues.

In [ ]:
async def repondre_avec_interruption(interrompre_apres_ms):
    # Votre code ici : renvoyer {"frames_joues": ..., "frames_apres": ..., "phrases_dites": [...]}
    return None

async def tester_interruption(delai):
    r = await repondre_avec_interruption(delai)
    if r is None:
        print("À faire : écrivez repondre_avec_interruption.")
        return
    print(f"Interruption à {delai} ms : {r['frames_joues']} frames lues, {r['frames_apres'] - r['frames_joues']} frame(s) après l'annulation, "
          f"{len(r['phrases_dites'])} phrase(s) entièrement dite(s).")
    assert r["frames_apres"] == r["frames_joues"], "des frames sont encore lues après l'annulation"

for delai in (900, 1500, 3000):
    await tester_interruption(delai)

Ce que l'exercice laisse de côté, et que le module 9 traitera : **que s'est-il réellement dit ?** Si Awa était à la moitié de sa deuxième phrase, l'**historique** envoyé au LLM au tour suivant doit contenir ce qui a été **entendu**, pas ce qui avait été **prévu**. Sinon, elle prétendra avoir annoncé un prix que le prospect n'a jamais entendu.

### 8.7 Instrumenter Awa : lire les mesures de chaque tour

Dans LiveKit Agents, chaque message de la conversation porte un champ `metrics` avec les latences du tour. D'après la documentation, pour un message **assistant** : `e2e_latency` (de la fin de la parole du prospect au début de la réponse), `llm_node_ttft` et `tts_node_ttfb`. Pour un message **prospect** : `end_of_turn_delay` et `transcription_delay`. Les deux premiers du LLM et du TTS ne sont renseignés que dans le pipeline STT, LLM, TTS (pas avec un modèle speech-to-speech).

Le code d'instrumentation (à mettre dans `awa_agent.py`, atelier) ressemble à ceci :

```python
@session.on("conversation_item_added")
def on_item(ev):
    if not isinstance(ev.item, ChatMessage):
        return
    m = ev.item.metrics
    if ev.item.role == "assistant" and m.get("e2e_latency") is not None:
        journal.append({"e2e": m["e2e_latency"], "ttft": m.get("llm_node_ttft"), "ttfb": m.get("tts_node_ttfb")})
```

Ici, on **simule** vingt tours (valeurs fictives, repère indicatif) pour s'entraîner à les résumer : une **médiane** et un **p95** par brique, comme au module 6.

In [ ]:
def tours_simules(n=20, graine=11):
    """Imite les mesures par tour : valeurs en secondes, comme les champs `metrics` de LiveKit (fictives)."""
    rng = np.random.default_rng(graine)
    tours = []
    for _ in range(n):
        fin_tour = 0.25 * rng.lognormal(0, 0.30)
        transcr = 0.10 * rng.lognormal(0, 0.35)
        ttft = 0.30 * rng.lognormal(0, 0.40)
        ttfb = 0.20 * rng.lognormal(0, 0.30)
        tours.append({"end_of_turn_delay": fin_tour, "transcription_delay": transcr,
                      "llm_node_ttft": ttft, "tts_node_ttfb": ttfb,
                      "e2e_latency": fin_tour + transcr + ttft + ttfb + 0.15 * rng.lognormal(0, 0.2)})
    return tours

TOURS = tours_simules()
print(f"{len(TOURS)} tours simulés. Exemple :", {k: round(v, 3) for k, v in TOURS[0].items()})

**[Guidé] Exercice 8.5** : écrivez `resume_metriques(tours)`, qui renvoie, pour chaque champ, un dictionnaire `{"mediane": ..., "p95": ...}` **en millisecondes**.

*Indice* : `np.median(v)` et `np.percentile(v, 95)`, sur les valeurs de ce champ dans tous les tours, multipliées par 1000.

In [ ]:
def resume_metriques(tours):
    resume = {}
    for champ in tours[0]:
        valeurs = [t[champ] * 1000 for t in tours]
        resume[champ] = {"mediane": 0.0, "p95": 0.0}   # TODO : médiane et p95 de `valeurs`
    return resume

R = resume_metriques(TOURS)
print(f"{'champ':22} {'médiane':>9} {'p95':>8}")
for champ, r in R.items():
    print(f"{champ:22} {r['mediane']:7.0f} ms {r['p95']:6.0f} ms")

In [ ]:
# Où se situe-t-on par rapport au budget d'environ 1 s ? (comparaison de la médiane et du p95 de e2e_latency)
cible = sum(BUDGET_MS.values())
e2e = R["e2e_latency"]
print(f"e2e_latency : médiane {e2e['mediane']:.0f} ms, p95 {e2e['p95']:.0f} ms, cible {cible} ms")
if e2e["mediane"] == 0:
    print("Complétez d'abord l'exercice 8.5.")
else:
    print("La médiane tient dans le budget." if e2e["mediane"] <= cible else "La médiane dépasse le budget.")
    print("Le p95 tient dans le budget." if e2e["p95"] <= cible else "Le p95 dépasse le budget : 1 appel sur 20 environ dérape.")

**Médiane et p95** : la médiane dit l'expérience **typique**, le p95 dit l'expérience des **mauvais jours**. Un agent dont la médiane est bonne et le p95 mauvais paraît « capricieux » : le prospect ne s'en souvient que pour les silences longs. On vise les deux, et on regarde d'abord le **p95** pour trouver ce qui dérape.

### À retenir

- Un tour d'Awa traverse cinq maillons : **détection de fin de tour, STT, LLM, TTS, réseau**. Le silence entendu est leur **somme**.
- On se donne un **budget de latence par brique**, puis on le **mesure**. Le maillon qui dépasse est celui qu'on traite en premier.
- `asyncio` **partage le temps d'attente** : `gather` pour lancer en parallèle, `Queue` pour relier les briques. Un calcul lourd bloque tout : on le déporte.
- Le **streaming de phrase en phrase** fait gagner la durée de génération de la suite de la réponse ; la **génération préemptive** parie sur la fin de tour.
- Annuler une réponse, c'est **annuler les tâches** et se souvenir de **ce qui a été réellement dit** (module 9).
- On **instrumente** Awa : `e2e_latency`, `llm_node_ttft`, `tts_node_ttfb` par tour, résumés en **médiane et p95**.

---
### Quiz du Module 8

**Q1.** Qu'est-ce que le « silence » que le prospect entend entre la fin de sa phrase et la voix d'Awa ?
a) uniquement le TTFT du LLM · b) uniquement le TTFB du TTS · c) uniquement le délai réseau · d) la somme des maillons : détection de fin de tour, STT, LLM, TTS, réseau et lecture

**Q2.** Que fait `asyncio.gather(agenda(), phrase())` ?
a) il exécute les deux sur deux cœurs différents · b) il exécute les deux coroutines en même temps et rend les deux résultats quand tout est fini · c) il exécute `phrase()` seulement si `agenda()` échoue · d) il convertit les coroutines en threads

**Q3.** Dans un pipeline relié par des files, pourquoi la brique TTS peut-elle parler avant que le LLM ait fini ?
a) parce que le TTS devine la suite · b) parce que le LLM envoie chaque phrase dans la file dès qu'elle est complète, et le TTS la lit aussitôt · c) parce que le TTS est plus rapide que le LLM · d) parce que `asyncio` utilise plusieurs cœurs

**Q4.** Le prospect coupe la parole pendant qu'Awa parle. Que faut-il faire côté code ?
a) annuler les tâches de génération et de lecture, vider les files, et garder dans l'historique ce qui a été réellement dit · b) laisser Awa finir sa phrase · c) redémarrer la session · d) envoyer un message d'erreur au prospect

**Q5.** Pourquoi regarder le p95 de la latence de bout en bout, en plus de la médiane ?
a) parce que le p95 est toujours plus petit que la médiane · b) parce que la médiane est une mesure fausse · c) parce que le p95 décrit les mauvais jours, qui sont ceux dont le prospect se souvient · d) parce que le p95 remplace le budget

In [ ]:
mes_reponses_m8 = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses

In [ ]:
def corriger_quiz_module8(reponses):
    attendu = {1: "d", 2: "b", 3: "b", 4: "a", 5: "c"}
    explications = {
        1: "Le silence entendu est la somme de tous les maillons : on le maîtrise avec un budget par brique.",
        2: "gather lance les coroutines ensemble et partage le temps d'attente. Ce n'est pas du calcul multi-cœur.",
        3: "Les files relient les briques : dès qu'une phrase est complète, elle part au TTS, pendant que le LLM écrit la suite.",
        4: "On annule les tâches, on vide les files et on garde dans l'historique ce qui a été entendu, pas ce qui était prévu.",
        5: "La médiane dit l'expérience typique, le p95 dit les mauvais jours : ceux qui font la réputation de l'agent.",
    }
    score = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score}/5")

corriger_quiz_module8(mes_reponses_m8)

---
## 9. Tours de parole et interruptions : la patience d'Awa

**Objectifs**
1. Comprendre le **rythme** d'une conversation humaine (environ 200 ms entre deux tours) et ce qu'il impose à un agent.
2. Voir ce que fait un **VAD** et régler son **seuil** et sa **durée de silence**.
3. Distinguer une **pause d'hésitation** d'une **fin de tour** avec une règle **sémantique** (phrases incomplètes, numéros dictés).
4. Classer ce que dit le prospect pendant qu'Awa parle : **interruption**, **backchannel** ou **fausse alerte**.
5. Mettre à jour le **contexte** après une interruption : ce qui a été **entendu**, pas ce qui était prévu.
6. Régler la **patience** d'Awa face à un prospect qui hésite ou qui change d'avis.

**Durée** : 40 minutes d'exercices, puis l'atelier (25 minutes) et le quiz (10 minutes). Tout fonctionne **hors ligne** : les signaux sont synthétiques.

**Attention aux chiffres** : les signaux, durées de pause et résultats de ce notebook sont **fictifs** (repère indicatif). Ils servent à comprendre la mécanique ; vos réglages réels se décident sur de vrais enregistrements (module 12).

In [ ]:
# Cellule de reprise du Module 9
import re, time, json, asyncio
import numpy as np, matplotlib.pyplot as plt
try:
    NAVY
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")
RED = "#C0392B"
FRAME_MS = 30        # une « trame » du VAD : quelques dizaines de ms (repère indicatif)
print("Module 9 prêt.")

### 9.1 Le rythme d'une conversation humaine

Quand deux personnes se parlent, le silence entre deux tours est **très court**. Stivers et ses collègues (2009) ont comparé **dix langues** et trouvé que toutes évitent le chevauchement et gardent un silence court entre les tours, avec une moyenne qui varie d'une langue à l'autre dans une fourchette d'environ 250 ms autour de la moyenne commune. Levinson et Torreira (2015) résument l'ordre de grandeur : des écarts d'environ **200 ms**, alors que préparer une réponse demande **plus de 600 ms**. Conséquence : un interlocuteur humain **anticipe la fin** du tour de l'autre et prépare sa réponse **avant** qu'elle arrive.

Un agent vocal ne peut pas faire pareil sans risque : il ne connaît la fin du tour qu'**après** un silence. Les études ci-dessus portent sur la conversation humaine ; elles ne donnent pas un « bon délai » pour un agent, mais un **ordre de grandeur** à garder en tête.

In [ ]:
HUMAIN_MS = 200            # ordre de grandeur des écarts entre tours (Stivers et al. 2009 ; Levinson et Torreira 2015)
BUDGET_AGENT_MS = 1000     # budget d'Awa (repère indicatif, module 8)
print(f"Écart humain typique : environ {HUMAIN_MS} ms. Budget d'Awa : {BUDGET_AGENT_MS} ms, soit {BUDGET_AGENT_MS // HUMAIN_MS} fois plus.")
print("Autrement dit : Awa ne pourra pas répondre aussi vite qu'un humain. Elle doit surtout éviter les deux fautes opposées,")
print("couper la parole (répondre trop tôt) et laisser un silence gênant (répondre trop tard).")

### 9.2 Le VAD : voir la parole image par image

Un **VAD** (*voice activity detection*) donne, pour chaque **trame** audio (quelques dizaines de millisecondes), une **probabilité de parole**. Silero VAD est celui que LiveKit utilise dans son plugin (`silero.VAD.load()`). Ses réglages principaux, d'après le plugin :

| Paramètre | Rôle | Défaut du plugin |
|---|---|---|
| `activation_threshold` | probabilité au-dessus de laquelle la parole **commence** | 0,5 |
| `deactivation_threshold` | probabilité en dessous de laquelle on considère que ce n'est **plus** de la parole | `max(seuil - 0,15 ; 0,01)` |
| `min_speech_duration` | durée minimale de parole pour la compter | 0,05 s |
| `min_silence_duration` | silence après lequel la parole est déclarée **terminée** | 0,55 s |

Le VAD ne **comprend** rien : il détecte de la voix, pas la fin d'une idée. Nous allons le **simuler** avec des signaux synthétiques (valeurs fictives), puis régler ses paramètres.

Un scénario est une suite de **morceaux** de phrase séparés par des **pauses**, par exemple un prospect qui hésite.

In [ ]:
def construire_signal(morceaux, pauses_ms, graine=0, douce=False, silence_final_ms=3000):
    """Fabrique une probabilité de parole par trame (valeurs fictives) et la vérité terrain.

    morceaux : liste de textes ; pauses_ms : pauses entre morceaux (len = len(morceaux) - 1).
    douce : voix faible (la probabilité de parole est plus basse)."""
    rng = np.random.default_rng(graine)
    probas, bornes, t = [], [], 0
    niveau = 0.55 if douce else 0.90
    for i, texte in enumerate(morceaux):
        duree = max(600, 280 * len(texte.split()))
        n = duree // FRAME_MS
        p = np.clip(rng.normal(niveau, 0.05 if douce else 0.07, n), 0, 1)
        creux = rng.random(n) < 0.04                       # petits creux de probabilité dans la parole
        p[creux] = rng.uniform(0.25, 0.5, creux.sum())
        probas.extend(p); bornes.append((t, t + n * FRAME_MS)); t += n * FRAME_MS
        pause = pauses_ms[i] if i < len(morceaux) - 1 else silence_final_ms
        m = pause // FRAME_MS
        q = np.clip(rng.normal(0.08, 0.05, m), 0, 1)
        bruit = rng.random(m) < 0.004                      # souffle, bruit de fond (rare)
        q[bruit] = rng.uniform(0.36, 0.45, bruit.sum())
        probas.extend(q); t += m * FRAME_MS
    return {"probas": np.array(probas), "bornes": bornes, "morceaux": morceaux, "pauses": list(pauses_ms),
            "fin_vraie": bornes[-1][1]}

demo = construire_signal(["Je voudrais un rendez-vous pour", "euh la semaine prochaine."], [900], graine=1)
x = np.arange(len(demo["probas"])) * FRAME_MS
plt.figure(figsize=(8, 2.8))
plt.plot(x, demo["probas"], color=TEAL, lw=1.2)
for seuil, c in ((0.5, AMBER), (0.35, GRAY)):
    plt.axhline(seuil, color=c, ls="--", lw=1)
plt.text(x[-1] * 0.62, 0.52, "seuil d'activation 0,5", color=AMBER, fontsize=9)
plt.text(x[-1] * 0.62, 0.37, "seuil de désactivation 0,35", color="#7A7F8C", fontsize=9)
plt.xlabel("ms"); plt.ylabel("probabilité de parole"); plt.title("Un prospect qui hésite : deux morceaux séparés par 900 ms", color=NAVY)
plt.tight_layout(); plt.show()

**[Guidé] Exercice 9.1** : écrivez `detecter_segments(probas, seuil, silence_min_ms)`, qui renvoie la liste des **segments de parole** `(début_ms, fin_ms)` et, pour chacun, l'instant où la **fin de tour est déclarée** (fin du segment + `silence_min_ms`).

Règles : la parole **commence** quand la probabilité atteint `seuil` ; elle reste en cours tant que la probabilité est **au-dessus du seuil de désactivation** (`max(seuil - 0,15 ; 0,01)`) ; elle **se termine** quand la probabilité reste **sous** ce seuil pendant `silence_min_ms`.

*Indice* : gardez un compteur de silence en ms. Une trame au-dessus du seuil de désactivation remet le compteur à zéro et mémorise l'instant de la dernière parole.

In [ ]:
def detecter_segments(probas, seuil=0.5, silence_min_ms=550):
    """Renvoie une liste de (debut_ms, fin_ms, declare_ms)."""
    desactivation = max(seuil - 0.15, 0.01)
    segments = []
    # TODO : parcourir les trames ; ouvrir un segment quand p >= seuil ;
    #        le fermer quand p < desactivation pendant silence_min_ms
    return segments

segs = detecter_segments(demo["probas"], 0.5, 550)
print("Segments détectés (début, fin, fin de tour déclarée) :", [(int(a), int(b), int(c)) for a, b, c in segs] or "aucun")

Avec 550 ms de silence minimum et une pause d'hésitation de 900 ms, le VAD déclare **deux** tours : il a **coupé** le prospect au milieu de sa phrase. C'est exactement le problème du module 5 (endpointing), vu côté audio.

### 9.3 Faire varier le seuil et le silence

On génère un petit **corpus** de vingt et un scénarios (valeurs fictives) : des phrases nettes, des prospects qui hésitent, des numéros dictés avec des pauses, deux voix douces et un cas de **pause très longue** (1,8 s). Pour chaque réglage, on mesure :

- les **coupures** : le pourcentage de scénarios où une fin de tour est déclarée **avant** la vraie fin de la parole ;
- l'**attente** : le délai médian entre la vraie fin de la parole et la fin de tour déclarée ;
- la **parole manquée** : le pourcentage de scénarios où le VAD n'entend pas la parole du tout ;
- les **faux déclenchements** : le pourcentage de scénarios où du bruit est pris pour de la parole **après** la vraie fin.

In [ ]:
def construire_corpus():
    rng = np.random.default_rng(42)
    sc = []
    nets = ["Oui, mardi à dix heures.", "Je m'appelle Mamadou Diop.", "Quelles sont vos offres ?", "Nous sommes huit salariés.",
            "Mon budget est de vingt-cinq mille francs.", "Non merci, pas pour le moment.", "Je voudrais un rendez-vous mardi.", "C'est trop cher pour nous."]
    for i, t in enumerate(nets):
        sc.append({"type": "net", **construire_signal([t], [], graine=i)})
    hesitants = [(["Je voudrais un rendez-vous pour", "euh la semaine prochaine."]),
                 (["Mon budget c'est environ", "disons vingt-cinq mille francs."]),
                 (["Nous avons une quincaillerie et", "enfin une petite entreprise."]),
                 (["Alors je pense que", "mardi ce serait bien."]),
                 (["Ce qui m'intéresse surtout c'est", "la gestion des stocks."]),
                 (["Je ne sais pas si", "oui, ça m'intéresse."])]
    for i, m in enumerate(hesitants):
        sc.append({"type": "hésitant", **construire_signal(m, [int(rng.integers(700, 1250))], graine=10 + i)})
    numeros = [["Mon numéro c'est le 77 616", "06 16."], ["Mon numéro est le 77", "616", "06 16."],
               ["Mon e-mail c'est fatou point sow", "arobase baobab point sn."], ["Le 12 novembre", "à 14 heures 30."]]
    for i, m in enumerate(numeros):
        sc.append({"type": "numéro", **construire_signal(m, [int(rng.integers(900, 1250)) for _ in m[:-1]], graine=20 + i)})
    for i, t in enumerate(["Oui, mardi à dix heures.", "Je m'appelle Aïssatou Ba."]):
        sc.append({"type": "voix douce", **construire_signal([t], [], graine=30 + i, douce=True)})
    sc.append({"type": "pause longue", **construire_signal(["Je réfléchis à", "mardi, finalement."], [1800], graine=40)})
    return sc

CORPUS = construire_corpus()
from collections import Counter
print(len(CORPUS), "scénarios :", dict(Counter(s["type"] for s in CORPUS)))

In [ ]:
def evaluer(seuil, silence_min_ms, corpus=CORPUS):
    """Pourcentages de coupures, de parole manquée et de faux déclenchements, et attente médiane (ms)."""
    coupures, attentes, manquees, faux = 0, [], 0, 0
    for s in corpus:
        segs = detecter_segments(s["probas"], seuil, silence_min_ms)
        reels = [g for g in segs if g[0] < s["fin_vraie"]]          # segments qui commencent avant la vraie fin
        faux += len(segs) > len(reels)                              # du bruit pris pour de la parole après la fin
        if not reels:
            manquees += 1
        elif len(reels) > 1:                                        # la parole a été découpée : le prospect est coupé
            coupures += 1
        else:
            attentes.append(reels[0][2] - s["fin_vraie"])
    n = len(corpus)
    return {"coupures": 100 * coupures / n, "attente": float(np.median(attentes)) if attentes else float("nan"),
            "manquees": 100 * manquees / n, "faux": 100 * faux / n}

print(f"{'silence min':>12} {'seuil':>6} {'coupures':>9} {'attente':>9} {'manquées':>9} {'faux décl.':>11}")
for silence in (300, 550, 800, 1300):
    for seuil in (0.35, 0.5, 0.7):
        r = evaluer(seuil, silence)
        print(f"{silence:9d} ms {seuil:6.2f} {r['coupures']:8.0f} % {r['attente']:6.0f} ms {r['manquees']:8.0f} % {r['faux']:9.0f} %")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
silences = [200, 300, 450, 550, 800, 1000, 1300, 1600]
for seuil, c in ((0.35, LIGHT), (0.5, TEAL), (0.7, AMBER)):
    res = [evaluer(seuil, s) for s in silences]
    axes[0].plot(silences, [r["coupures"] for r in res], "o-", color=c, label=f"seuil {seuil}")
    axes[1].plot([r["attente"] for r in res], [r["coupures"] for r in res], "o-", color=c, label=f"seuil {seuil}")
axes[0].set_xlabel("silence minimum (ms)"); axes[0].set_ylabel("coupures (%)")
axes[1].set_xlabel("attente médiane (ms)"); axes[1].set_ylabel("coupures (%)")
axes[0].legend(frameon=False, fontsize=8); axes[0].set_title("Plus on attend, moins on coupe", color=NAVY, fontsize=10)
axes[1].set_title("Le compromis : attente contre coupures", color=NAVY, fontsize=10)
plt.tight_layout(); plt.show()

Trois lectures à retenir :
- **Le silence minimum** règle le compromis : l'allonger réduit les coupures mais ajoute **exactement** ce délai à chaque réponse, même quand le prospect a vraiment fini.
- **Le seuil d'activation** agit sur la **sensibilité** : un seuil haut (0,7) **manque** les voix douces, un seuil bas (0,35) laisse passer plus de bruit. Pour une voix douce, on baisse le seuil et on vérifie qu'on n'a pas créé de fausses détections.
- Aucun réglage ne supprime les coupures **sans** payer en attente. Pour sortir de ce compromis, il faut **comprendre** ce que dit le prospect.

### 9.4 Phrases incomplètes : la fin de tour sémantique

Une pause après « Je voudrais un rendez-vous pour... » n'est pas une fin de tour, une pause après « Oui, mardi à dix heures. » en est une. Un **détecteur de fin de tour sémantique** regarde le **texte** (ou l'audio : intonation, rythme) pour estimer si l'idée est terminée. LiveKit propose ce type de modèle en plus du VAD : le modèle **audio** `inference.TurnDetector` est recommandé (il couvre 14 langues dont le français), l'ancien modèle de **texte** `MultilingualModel` est déprécié.

Ici, on écrit une **règle** simple, pour comprendre le principe et ses limites. Une phrase est **incomplète** si elle :
- finit par un **mot de liaison** ou une préposition (« et », « pour », « que », « c'est », « environ », etc.) ;
- finit par une **hésitation** (« euh », « hum », « alors ») ;
- contient un **numéro dicté inachevé** (moins de neuf chiffres pour un numéro de téléphone après « numéro »).

In [ ]:
MOTS_LIAISON = {"et", "ou", "mais", "donc", "que", "qui", "pour", "de", "du", "des", "le", "la", "les", "un", "une", "à", "au", "aux",
                "avec", "sans", "dans", "sur", "par", "c'est", "environ", "disons", "alors", "enfin", "parce", "comme", "si", "je", "mon", "ma", "mes"}
HESITATIONS = {"euh", "hum", "heu", "bah", "ben", "alors"}

PHRASES_TEST = [
    ("Je voudrais un rendez-vous pour", False),
    ("Oui, mardi à dix heures.", True),
    ("Mon budget c'est environ", False),
    ("Quelles sont vos offres ?", True),
    ("Je voudrais un rendez-vous euh", False),
    ("Mon numéro c'est le 77 616", False),
    ("Mon numéro c'est le 77 616 06 16", True),
    ("Nous sommes huit salariés", True),
    ("Je ne sais pas si", False),
    ("Non merci", True),
]
print(len(PHRASES_TEST), "phrases de test (texte, complète ?)")

**[Guidé] Exercice 9.2** : écrivez `phrase_complete(texte)`, qui renvoie `True` si la phrase peut être considérée comme terminée.

*Indice* : mettez le texte en minuscules, retirez la ponctuation finale (`. , ? ! ...`), prenez le **dernier mot**. Pour les numéros : si le texte contient « numéro », comptez les chiffres avec `len(re.findall(r"\d", texte))`.

In [ ]:
def phrase_complete(texte):
    t = texte.lower().strip()
    # TODO : numéro dicté inachevé (le texte contient « numéro » et moins de 9 chiffres) -> False
    # TODO : dernier mot dans MOTS_LIAISON ou HESITATIONS -> False
    return True

bons = 0
for texte, attendu in PHRASES_TEST:
    ok = phrase_complete(texte) == attendu
    bons += ok
    print(("ok       " if ok else "À REVOIR  ") + f"{texte!r:42} -> {phrase_complete(texte)}")
print(f"\n{bons}/{len(PHRASES_TEST)} corrects")

Cette règle **échoue** sur des cas qu'on devine facilement : « Je voudrais un rendez-vous » (finit par un nom, mais semble inachevé sans date), ou une phrase complète terminée par « ... pour moi » (« pour » n'est pas le dernier mot, tout va bien) contre « merci pour » (mot de liaison, inachevé). Un modèle appris regarde plus que le dernier mot. L'intérêt de la règle est de **voir le principe** : le texte décide de la **patience**.

### 9.5 [Avancé] Combiner le VAD et la règle sémantique

Dans la suite de l'**endpointing adaptatif du module 5** (0,5 s si la phrase est complète, 1,3 s si elle est inachevée), on règle maintenant la **durée de silence** selon le **texte**.

**[Avancé] Exercice 9.3** : écrivez `delai_semantique(texte)`, qui renvoie le silence (en ms) à attendre **avant de déclarer la fin de tour**, selon ce que le prospect vient de dire. Puis lancez la comparaison : le corpus rejoue chaque scénario avec le **texte cumulé** à chaque pause.

Contraintes : phrase complète 500 ms ; phrase incomplète 1 300 ms ; vous pouvez ajouter un cas **numéro inachevé** plus patient (par exemple 1 500 ms). Comparez avec le silence fixe de 550 ms et celui de 1 150 ms du module 5.

In [ ]:
def delai_semantique(texte):
    # Votre code ici : renvoyer le silence à attendre, en ms, selon le texte entendu jusqu'ici
    return 550

def simuler_decision(scenario, delai_fn):
    """Rejoue un scénario : à chaque pause, on compare sa durée au délai voulu pour le texte cumulé.
    Renvoie (coupure ?, attente après la vraie fin en ms)."""
    segs = detecter_segments(scenario["probas"], 0.5, 300)       # segmentation fine : une pause d'au moins 300 ms
    if len(segs) != len(scenario["morceaux"]):
        return None
    texte = ""
    for i, (d, f, _) in enumerate(segs):
        texte = (texte + " " + scenario["morceaux"][i]).strip()
        pause = (segs[i + 1][0] - f) if i + 1 < len(segs) else 10**9
        delai = delai_fn(texte)
        if pause >= delai:                                      # on déclare la fin de tour pendant cette pause
            if i + 1 < len(segs):
                return True, 0                                  # coupure : le prospect allait reprendre
            return False, (f + delai) - scenario["fin_vraie"]

def comparer(delai_fn, corpus=CORPUS):
    coupures, attentes, valides, cas = 0, [], 0, []
    for s in corpus:
        r = simuler_decision(s, delai_fn)
        if r is None:
            continue
        valides += 1
        if r[0]:
            coupures += 1
            cas.append(" / ".join(s["morceaux"]))
        else:
            attentes.append(r[1])
    return {"coupures": 100 * coupures / max(valides, 1), "attente": float(np.median(attentes)) if attentes else float("nan"),
            "valides": valides, "cas": cas}

print(f"{'stratégie':26} {'coupures':>9} {'attente médiane':>16}")
for nom, fn in (("VAD seul, silence 550 ms", lambda t: 550), ("VAD seul, silence 1150 ms", lambda t: 1150),
                ("VAD + règle sémantique", delai_semantique)):
    r = comparer(fn)
    print(f"{nom:26} {r['coupures']:7.0f} % {r['attente']:13.0f} ms")
    if fn is delai_semantique:
        print("   cas encore coupés :", r["cas"] or "aucun")

Ce que la comparaison montre : la règle sémantique **coupe beaucoup moins** que le silence de 550 ms, tout en répondant **plus vite** que le silence de 1 150 ms quand la phrase est complète (500 ms au lieu de 1 150 ms). Lisez la liste des **cas encore coupés** : la pause très longue (1,8 s, plus que 1,3 s) et un e-mail ou une date dictés en plusieurs morceaux (« ... fatou point sow », « le 12 novembre » **semblent** complets) échappent à la règle. Une règle écrite à la main se trompe dès que le français devient moins prévisible. D'où l'intérêt d'un **modèle appris** et d'une **mesure sur de vrais appels** (module 12).

### 9.6 Quand le prospect parle pendant qu'Awa parle

Trois cas très différents, et un seul doit **arrêter** Awa :

| Cas | Exemple | Que faire |
|---|---|---|
| **Interruption** | « Attendez, je n'ai pas compris » | s'arrêter, écouter |
| **Backchannel** | « hum », « d'accord », « oui » | continuer : le prospect signale qu'il écoute |
| **Fausse alerte** | une toux, du bruit, un mot isolé non transcrit | continuer (et reprendre si Awa s'était arrêtée) |

LiveKit propose pour cela des options d'`InterruptionOptions` (`min_duration` à 0,5 s par défaut, `min_words`, un mode **adaptatif** qui distingue interruption et backchannel avec un modèle, et la **reprise après fausse interruption**). Ici, une règle simple, pour comprendre.

**[Guidé] Exercice 9.4** : écrivez `classer_chevauchement(texte, duree_ms, min_duree_ms=500, min_mots=1)`, qui renvoie `"fausse alerte"`, `"backchannel"` ou `"interruption"`.

Règles : **aucun mot** transcrit : fausse alerte. **Tous** les mots dans `BACKCHANNELS` : backchannel. Sinon, c'est une **interruption** si la durée atteint `min_duree_ms` **et** que le nombre de mots atteint `min_mots` ; sinon fausse alerte.

In [ ]:
BACKCHANNELS = {"oui", "hum", "mm", "mmh", "ok", "d'accord", "ah", "je vois", "bien sûr", "voilà", "ouais", "mm-hm", "hmm"}

def classer_chevauchement(texte, duree_ms, min_duree_ms=500, min_mots=1):
    # TODO : « fausse alerte », « backchannel » ou « interruption » selon les règles ci-dessus
    return "interruption"

CHEVAUCHEMENTS = [("hum", 300, "backchannel"), ("oui", 400, "backchannel"), ("d'accord", 600, "backchannel"),
                  ("attendez, je n'ai pas compris", 1800, "interruption"), ("", 700, "fausse alerte"),
                  ("non, plutôt mardi", 1500, "interruption"), ("mm-hm", 350, "backchannel"), ("c'est trop cher", 1400, "interruption"),
                  ("pardon", 200, "fausse alerte")]
bons = 0
for texte, duree, attendu in CHEVAUCHEMENTS:
    r = classer_chevauchement(texte, duree)
    bons += r == attendu
    print(("ok       " if r == attendu else "À REVOIR  ") + f"{texte!r:34} {duree:5d} ms -> {r}")
print(f"\n{bons}/{len(CHEVAUCHEMENTS)} corrects")

Remarquez « pardon » (200 ms) : un mot **peut** être une vraie prise de parole, mais trop court pour être fiable. Régler `min_duree_ms` et `min_mots` est un **compromis** : trop haut, le prospect doit crier pour couper Awa ; trop bas, chaque « hum » la fait taire. C'est ce que règle l'atelier.

### 9.7 Mettre à jour le contexte après une interruption

Awa avait prévu : « Nous avons trois offres. Essentiel à vingt-cinq mille francs CFA, Pro à quarante-neuf mille francs CFA, et Entreprise sur devis. » Le prospect coupe au milieu. Si l'**historique** garde la réponse **complète**, Awa croira avoir annoncé le prix du Pro, que le prospect n'a **jamais entendu**.

D'après la documentation LiveKit, après une interruption l'agent **tronque** son historique à la partie de la parole **que l'utilisateur a entendue**. Le message d'assistant porte aussi un drapeau `interrupted`. Ici, on le fait à la main, à partir de la durée **réellement jouée**.

**[Guidé] Exercice 9.5** : écrivez `texte_entendu(reponse, ms_joues, ms_par_mot=250)`, qui renvoie les **mots entendus** : autant de mots que la durée jouée en contient (`ms_joues // ms_par_mot`).

*Indice* : `reponse.split()[:n]`, puis `" ".join(...)`.

In [ ]:
REPONSE_PREVUE = ("Nous avons trois offres. Essentiel à vingt-cinq mille francs CFA, Pro à quarante-neuf mille francs CFA, "
                  "et Entreprise sur devis.")

def texte_entendu(reponse, ms_joues, ms_par_mot=250):
    # TODO : ne garder que les mots réellement joués
    return reponse

historique = [{"role": "user", "content": "Quelles sont vos offres ?"}]
heard = texte_entendu(REPONSE_PREVUE, 2600)
historique.append({"role": "assistant", "content": heard, "interrupted": True})
print("Prévu  :", REPONSE_PREVUE)
print("Entendu:", heard)
if "quarante-neuf" in heard:
    print("\nÀ revoir : l'historique contient un prix que le prospect n'a pas entendu.")
else:
    print("\nL'historique ne contient que ce que le prospect a entendu.")

Deux détails de réalisme : on **coupe au mot**, jamais au milieu d'un mot ; et on **garde le drapeau** `interrupted`, qui permet de journaliser les interruptions (module 12) et d'ajouter au prompt « si ton message précédent est marqué interrompu, ne présume pas que le prospect a tout entendu ».

### 9.8 Atelier : régler la patience d'Awa

Trois **profils de patience**, avec le réglage LiveKit correspondant (à lire, non exécuté ici) :

| Profil | Silence avant fin de tour | Pour qui |
|---|---|---|
| **Vive** | 0,3 s à 1,5 s (`min_delay`, `max_delay`) | prospect qui répond net, ligne bruyante |
| **Équilibrée** | 0,5 s à 3,0 s (valeurs par défaut) | usage général |
| **Patiente** | 0,9 s à 4,0 s | prospect qui hésite, numéros dictés |

Les profils se testent sur le corpus de ce notebook : un profil **patient** coupe moins mais répond plus lentement. À vous de choisir celui d'Awa pour **Mamadou** (net), **Aïssatou** (réfléchit longuement) et **Jean-Pierre** (change d'avis en cours de réponse).

In [ ]:
PROFILS = {"vive": 300, "équilibrée": 550, "patiente": 1000, "sémantique": None}

print(f"{'profil':12} {'coupures':>9} {'attente médiane':>16}")
for nom, silence in PROFILS.items():
    fn = delai_semantique if silence is None else (lambda t, s=silence: s)
    r = comparer(fn)
    print(f"{nom:12} {r['coupures']:7.0f} % {r['attente']:13.0f} ms")

CONFIG_LIVEKIT = {                      # à adapter ; vérifier les noms dans la documentation le jour J
    "endpointing": {"min_delay": 0.5, "max_delay": 3.0},
    "interruption": {"min_duration": 0.5, "min_words": 1, "mode": "adaptive",
                     "false_interruption_timeout": 2.0, "resume_false_interruption": True},
}
print("\nConfiguration de départ pour AgentSession(turn_handling=...) :")
print(json.dumps(CONFIG_LIVEKIT, indent=2, ensure_ascii=False))

**Questions pour l'atelier** (à répondre dans une cellule de texte) :
1. Quel profil choisissez-vous pour Awa, et **pourquoi** ? Donnez les deux chiffres sur lesquels vous vous appuyez.
2. Que se passe-t-il pour **Aïssatou** (pauses longues) et pour **Mamadou** (réponses nettes) avec votre choix ?
3. Quel **message** Awa peut-elle dire après une coupure pour la **réparer** (« Pardon, je vous ai coupé, je vous écoute ») ?
4. Quelle **interruption** faut-il **ne pas** laisser couper Awa (par exemple pendant l'annonce légale de l'IA, module 13) ?

### À retenir

- Entre deux tours, les humains laissent environ **200 ms** (Stivers et al. 2009 ; Levinson et Torreira 2015) et **anticipent** la fin du tour. Un agent ne peut pas : il attend un **silence**.
- Un **VAD** détecte de la voix, pas la **fin d'une idée**. Son **silence minimum** règle un compromis : plus long, moins de coupures, mais plus d'attente à chaque tour.
- Le **seuil d'activation** règle la sensibilité : trop haut, il manque les voix douces ; trop bas, il laisse passer le bruit.
- Une **règle ou un modèle sémantique** règle la **patience** selon le contenu : court si la phrase est complète, long si elle est inachevée (numéros, hésitations).
- Pendant qu'Awa parle, on distingue **interruption**, **backchannel** et **fausse alerte**. Seule la première l'arrête.
- Après une interruption, l'historique garde ce qui a été **entendu**, marqué `interrupted`.

---
### Quiz du Module 9

**Q1.** Quel est l'ordre de grandeur de l'écart entre deux tours dans une conversation humaine ?
a) environ 2 secondes · b) environ 600 ms · c) de l'ordre de 200 ms, avec une anticipation de la fin du tour · d) zéro, les gens parlent toujours en même temps

**Q2.** Pourquoi un VAD seul coupe-t-il souvent un prospect qui hésite ?
a) parce qu'il détecte le silence, pas la fin de l'idée : une pause d'hésitation ressemble à une fin de tour · b) parce qu'il comprend mal le français · c) parce que son seuil d'activation est trop bas · d) parce qu'il ne reçoit pas assez d'audio

**Q3.** Awa n'entend pas un prospect à voix très douce. Quel premier réglage essayer ?
a) augmenter le seuil d'activation · b) allonger le silence minimum · c) changer le modèle de voix du TTS · d) baisser le seuil d'activation, puis vérifier qu'on n'a pas créé de fausses détections

**Q4.** Le prospect dit « hum » pendant qu'Awa parle. Que doit faire Awa ?
a) s'arrêter immédiatement · b) continuer : c'est un backchannel, un signal d'écoute · c) poser une nouvelle question · d) signaler une erreur du STT

**Q5.** Awa est interrompue au milieu de sa deuxième phrase. Que contient l'historique envoyé au LLM au tour suivant ?
a) seulement la partie entendue du message, marquée comme interrompue · b) la réponse complète qui était prévue · c) rien du tout · d) la réponse prévue et une excuse

In [ ]:
mes_reponses_m9 = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses

In [ ]:
def corriger_quiz_module9(reponses):
    attendu = {1: "c", 2: "a", 3: "d", 4: "b", 5: "a"}
    explications = {
        1: "Les écarts typiques sont de l'ordre de 200 ms, alors que préparer une réponse demande bien plus : les humains anticipent la fin du tour.",
        2: "Le VAD détecte de la voix. Une pause d'hésitation est du silence comme une fin de tour : il faut du sens (texte ou intonation) pour les distinguer.",
        3: "On baisse le seuil d'activation pour mieux entendre, puis on contrôle les fausses détections avec des enregistrements réels.",
        4: "Un backchannel signale que le prospect écoute : couper Awa à chaque « hum » rendrait la conversation hachée.",
        5: "On tronque à ce qui a été entendu et on garde le drapeau interrupted : Awa ne doit pas croire avoir annoncé un prix non entendu.",
    }
    score = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score}/5")

corriger_quiz_module9(mes_reponses_m9)

---
## 10. Attente, objections, transfert et accents : une Awa fluide

**Objectifs**
1. Éviter le **silence** pendant un outil lent : **préambule**, **filler**, **relance** et **timeout**.
2. Choisir un filler **adapté au contexte**, sans **répétition**.
3. Répondre aux **objections** commerciales (« c'est cher », « envoyez-moi un e-mail ») sans inventer de remise.
4. Décider **quand transférer** vers un humain, et lui transmettre un **résumé**.
5. **Comparer des STT** sur les accents, avec le WER et les erreurs sur les entités.

**Durée** : 2 h 30 au total (voir le déroulé du deck). Environ 60 minutes d'exercices dans ce notebook, puis l'atelier et le quiz. Tout fonctionne **hors ligne** : l'agenda lent, les transcriptions et les profils d'erreur des STT sont **simulés**.

**Attention aux chiffres** : seuils, durées et taux d'erreur de ce notebook sont **fictifs** (repère indicatif). Ils servent à comprendre la mécanique. Vos réglages et votre choix de STT se décident sur **vos** enregistrements.

In [ ]:
# Cellule de reprise du Module 10
import re, time, json, asyncio, random
from difflib import SequenceMatcher
import numpy as np, matplotlib.pyplot as plt
try:
    NAVY
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")
RED = "#C0392B"
SEUILS = {"filler": 1.2, "relance": 4.0, "timeout": 8.0}   # secondes, repères indicatifs
print("Module 10 prêt. Seuils de départ :", SEUILS)

### 10.1 Le silence pendant un outil lent

Awa appelle l'agenda (`check_availability`). Si l'agenda met **5 secondes**, que se passe-t-il pour le prospect ? **Rien**, pendant 5 secondes. Au téléphone, un silence de quelques secondes fait penser que la ligne est coupée ou qu'Awa n'a pas compris (repère indicatif : c'est dès une à deux secondes que ça se remarque).

Quatre outils pour combler l'attente, du plus tôt au plus tard :

| Outil | Quand | Exemple |
|---|---|---|
| **Préambule** | avant l'appel de l'outil | « Je regarde l'agenda. » |
| **Filler** | après un court délai sans réponse | « Un instant, je consulte le planning. » |
| **Relance** | si l'attente se prolonge | « Je suis toujours dessus, merci de patienter. » |
| **Timeout** | au bout d'un délai maximal | « Je n'arrive pas à joindre l'agenda. Je peux vous rappeler. » |

Dans LiveKit Agents, un outil peut jouer une phrase de remplissage pendant qu'il travaille avec `async with context.with_filler(...)`. Cette phrase est dite directement (`session.say`) : elle **ne repasse pas par le LLM**, donc pas de latence ni de coût en plus (docstring de `RunContext.with_filler`, livekit-agents 1.8.10).

**[Guidé] Exercice 10.1** : écrivez `evenements_attente(duree, seuils)`. Elle reçoit la **durée de l'outil** en secondes et renvoie la liste des événements `(instant, nom)` qu'Awa produit :
- `"filler"` à l'instant `seuils["filler"]` si l'outil n'a pas répondu ;
- `"relance"` à l'instant `seuils["relance"]` si l'outil n'a toujours pas répondu ;
- `"timeout"` à l'instant `seuils["timeout"]` si l'outil n'a toujours pas répondu : **c'est la fin**, il n'y a pas de résultat ;
- `"resultat"` à l'instant `duree` si l'outil répond avant le timeout.

*Indice* : parcourez les trois seuils dans l'ordre. Dès que `duree` est **inférieure ou égale** à un seuil, ajoutez `("resultat", duree)` et arrêtez-vous. Sinon, ajoutez l'événement du seuil. Pour le timeout, ajoutez-le puis arrêtez-vous.

In [ ]:
def evenements_attente(duree, seuils):
    evenements = []
    # TODO : pour nom in ("filler", "relance", "timeout") :
    #   si duree <= seuils[nom] : ajouter ("resultat", duree) et renvoyer la liste
    #   sinon ajouter (seuils[nom], nom) ; au timeout, renvoyer la liste (pas de résultat)
    evenements.append((duree, "resultat"))
    return evenements

CAS_ATTENTE = {
    0.6: [(0.6, "resultat")],
    2.0: [(1.2, "filler"), (2.0, "resultat")],
    5.0: [(1.2, "filler"), (4.0, "relance"), (5.0, "resultat")],
    12.0: [(1.2, "filler"), (4.0, "relance"), (8.0, "timeout")],
}
bons = 0
for duree, attendu in CAS_ATTENTE.items():
    obtenu = evenements_attente(duree, SEUILS)
    ok = obtenu == attendu
    bons += ok
    print(("ok       " if ok else "À REVOIR  ") + f"outil de {duree:4.1f} s -> {obtenu}")
print(f"\n{bons}/{len(CAS_ATTENTE)} corrects")

Un cas de plus à regarder : le **silence perçu**. C'est le plus long intervalle sans parole d'Awa. Sans filler, un outil de 5 s laisse 5 s de silence ; avec les seuils ci-dessus, le plus long silence est de 2,8 s (entre 1,2 s et 4,0 s). Ces seuils ne suppriment pas l'attente, ils la **découpent**.

In [ ]:
def silence_max(duree, seuils):
    """Plus long intervalle sans parole d'Awa pendant l'attente (le préambule est dit à t = 0)."""
    instants = [0.0] + [t for t, nom in evenements_attente(duree, seuils)]
    if not any(nom == "timeout" for _, nom in evenements_attente(duree, seuils)):
        instants.append(duree)
    return max(b - a for a, b in zip(instants, instants[1:])) if len(instants) > 1 else 0.0

print("durée de l'outil -> plus long silence perçu (s)\n")
for d in (0.6, 2.0, 5.0, 12.0):
    print(f"  {d:4.1f} s -> {silence_max(d, SEUILS):.1f} s")
sans = {"filler": 99, "relance": 99, "timeout": 99}
print("\nsans aucun filler, un outil de 5 s -> silence de", silence_max(5.0, sans), "s")

### 10.2 Un outil lent en asyncio : filler, relance, timeout

On simule maintenant l'agenda lent avec `asyncio`. Le temps est **accéléré** (1 seconde simulée = 20 ms réelles) pour que le notebook reste rapide. Le principe est celui de `avec_attente` dans `awa_agent.py` : on **attend le résultat avec un délai** (`asyncio.wait`), et chaque fois que le délai expire, Awa dit quelque chose.

In [ ]:
ECHELLE = 0.02      # 1 seconde simulée = 20 ms réelles

async def agenda_lent(duree):
    await asyncio.sleep(duree * ECHELLE)
    return {"creneaux": ["09:00", "09:30", "14:30"]}

async def appeler_avec_attente(outil, seuils, dire):
    """Lance l'outil, puis dit filler, relance, timeout selon les seuils. Renvoie le résultat, ou None au timeout."""
    debut = time.perf_counter()
    tache = asyncio.ensure_future(outil)
    for nom in ("filler", "relance", "timeout"):
        reste = seuils[nom] * ECHELLE - (time.perf_counter() - debut)
        finies, _ = await asyncio.wait({tache}, timeout=max(reste, 0))
        if finies:
            return tache.result()
        dire(nom, (time.perf_counter() - debut) / ECHELLE)
        if nom == "timeout":
            tache.cancel()
            return None

async def essayer(duree):
    dits = []
    t0 = time.perf_counter()
    resultat = await appeler_avec_attente(agenda_lent(duree), SEUILS, lambda nom, t: dits.append((round(t, 1), nom)))
    fin = round((time.perf_counter() - t0) / ECHELLE, 1)
    return dits, resultat, fin

for duree in (0.6, 2.0, 5.0, 12.0):
    dits, resultat, fin = await essayer(duree)
    print(f"outil de {duree:4.1f} s -> dit : {[nom for _, nom in dits]} ; résultat : {'oui' if resultat else 'non (timeout)'}")
    attendus = [nom for _, nom in evenements_attente(duree, SEUILS) if nom != "resultat"]
    if [n for _, n in dits] != attendus:
        print("   À REVOIR : écart avec l'exercice 10.1, attendu", attendus)

**Le timeout n'est pas un échec muet.** Au timeout, Awa **annule** l'appel à l'outil et propose une **solution de repli** : rappeler le prospect, ou le passer à un conseiller. Dans `awa_agent.py`, l'outil lève `ToolError` avec ce message de secours et le LLM le reformule à l'oral.

**Tâches en arrière-plan.** Quand un outil est long mais pas bloquant (par exemple l'envoi d'un e-mail de confirmation), l'agent peut **continuer à parler** pendant qu'il travaille. Dans LiveKit Agents, un outil asynchrone passe en arrière-plan à son premier `await ctx.update(...)` : le prospect entend tout de suite un accusé de réception, et le résultat final est annoncé quand l'agent est libre (documentation LiveKit, « Async tools »). Pour **réserver un créneau**, en revanche, mieux vaut attendre le résultat : on ne confirme pas un rendez-vous avant de savoir s'il est pris.

### 10.3 Choisir le filler selon le contexte, sans répétition

Dire trois fois de suite « Un instant, s'il vous plaît » est pire que le silence : ça sonne **robotique**. Un bon filler :
- est **court** (quelques mots, environ 2 à 3 secondes de parole) ;
- **correspond à l'outil** (agenda, fiche prospect, calcul) ;
- n'est **pas répété** de façon rapprochée ;
- change quand c'est une **relance** (on ne dit pas « je regarde l'agenda » pour la deuxième fois).

In [ ]:
FILLERS = {
    "agenda": ["Je regarde l'agenda.", "Je vérifie les disponibilités.", "Un instant, je consulte le planning."],
    "crm": ["Je note tout cela.", "Je prépare votre fiche, un instant.", "J'enregistre vos informations."],
    "generique": ["Un instant, s'il vous plaît.", "Je m'en occupe tout de suite."],
    "relance": ["Je suis toujours dessus, merci de patienter.", "Le système est un peu lent, je n'ai pas oublié votre demande.",
                "Merci de votre patience, j'y suis presque."],
}
print({k: len(v) for k, v in FILLERS.items()}, "phrases par catégorie")

**[Avancé] Exercice 10.2** : écrivez `choisir_filler(outil, deja_dits, etape)`. La consigne est ouverte ; votre fonction doit :
1. renvoyer une phrase de `FILLERS` de la catégorie de l'**outil** à l'étape 0, et de la catégorie `"relance"` aux étapes suivantes ; un outil inconnu utilise `"generique"` ;
2. ne **jamais** renvoyer l'une des **trois dernières** phrases de `deja_dits` quand c'est possible ;
3. quand toutes les phrases de la catégorie ont été dites récemment, renvoyer **la moins récente** plutôt que de rester muette ;
4. **ajouter** la phrase choisie à `deja_dits`.

Les tests plus bas vérifient ces quatre points.

In [ ]:
def choisir_filler(outil, deja_dits, etape):
    # TODO : à vous d'écrire, selon la consigne ci-dessus
    return FILLERS["generique"][0]

# ---- tests ----
def tester_fillers():
    echecs = []
    # 1. catégorie
    d = []
    if choisir_filler("agenda", d, 0) not in FILLERS["agenda"]: echecs.append("étape 0 : catégorie de l'outil")
    if choisir_filler("agenda", d, 1) not in FILLERS["relance"]: echecs.append("étape 1 : catégorie relance")
    if choisir_filler("meteo", [], 0) not in FILLERS["generique"]: echecs.append("outil inconnu : generique")
    # 2. pas de répétition dans les trois dernières phrases
    d = []
    dits = [choisir_filler("crm", d, 0) for _ in range(6)]
    if len(set(dits[:3])) < 3 or any(dits[i] == dits[i - 1] for i in range(1, 6)): echecs.append("répétition rapprochée")
    # 3. catégorie épuisée : on prend la moins récente, jamais la toute dernière
    d = ["Un instant, s'il vous plaît.", "Je m'en occupe tout de suite."]
    p = choisir_filler("meteo", d, 0)
    if p != "Un instant, s'il vous plaît.": echecs.append("catégorie épuisée : la moins récente")
    # 4. historique mis à jour
    d = []
    p = choisir_filler("agenda", d, 0)
    if d != [p]: echecs.append("deja_dits doit être mis à jour")
    # court
    if any(len(x.split()) > 12 for liste in FILLERS.values() for x in liste): echecs.append("fillers trop longs")
    print("Tous les tests passent." if not echecs else "À REVOIR : " + " ; ".join(echecs))

tester_fillers()

**Cas particulier : quand le prospect parle pendant le filler.** Le filler est une parole d'Awa comme une autre : s'il **interrompt** (module 9), Awa se tait. Ne rendez pas un filler **ininterruptible** : le prospect qui dit « laissez tomber » doit pouvoir être entendu. Réservez `disallow_interruptions()` aux phrases qui ne doivent pas être coupées (par exemple l'annonce légale, module 13).

### 10.4 Les objections commerciales

Deux objections reviennent sans cesse :
- **« C'est cher. »** Awa ne **négocie pas** : elle n'a **aucune** remise à offrir, hors « deux mois offerts en paiement annuel ». Elle reconnaît l'objection, redonne le **tarif exact**, rappelle cette seule réduction, et propose un **rendez-vous** de 30 minutes.
- **« Envoyez-moi un e-mail. »** C'est souvent une manière polie de **clore**. Awa accepte, **relit l'adresse** lettre par lettre (module 7), et propose aussi le rendez-vous, sans insister.

Le danger : un LLM trop serviable peut **inventer** une remise pour garder le prospect. Un garde-fou sur la **sortie** d'Awa est donc utile, en plus du prompt.

**[Guidé] Exercice 10.3** : écrivez `classer_objection(texte)`. Elle renvoie l'une de ces étiquettes :
- `"prix"` si le texte parle de prix (mots : `cher`, `coûte`, `budget`, `prix`, `tarif`),
- `"email"` si le prospect demande un envoi écrit (`e-mail`, `mail`, `par écrit`),
- `"reflechir"` (`réfléchir`, `plus tard`, `j'y pense`, `pas maintenant`),
- `"concurrent"` (`concurrent`, `déjà un logiciel`, `j'utilise déjà`),
- `"aucune"` sinon.

*Indice* : mettez le texte en minuscules, et parcourez un dictionnaire `{étiquette: [mots]}` dans l'ordre (ici, `email` vient en premier). La **première** étiquette dont un mot apparaît gagne. Attention à `e-mail` écrit avec ou sans tiret.

In [ ]:
MOTS_OBJECTION = {
    "email": ["e-mail", "email", "mail", "par écrit"],
    "prix": ["cher", "coûte", "budget", "prix", "tarif"],
    "reflechir": ["réfléchir", "plus tard", "j'y pense", "pas maintenant"],
    "concurrent": ["concurrent", "déjà un logiciel", "j'utilise déjà"],
}

def classer_objection(texte):
    t = texte.lower()
    # TODO : parcourir MOTS_OBJECTION dans l'ordre ; renvoyer la première étiquette dont un mot est dans t
    return "aucune"

TESTS_OBJECTION = [
    ("C'est trop cher pour nous.", "prix"),
    ("Envoyez-moi un e-mail, je regarderai.", "email"),
    ("Envoyez-moi un mail avec les tarifs.", "email"),
    ("Je dois réfléchir, je vous rappelle.", "reflechir"),
    ("On utilise déjà un logiciel de gestion.", "concurrent"),
    ("Combien coûte l'offre Pro ?", "prix"),
    ("Oui, mardi à dix heures.", "aucune"),
]
bons = 0
for texte, attendu in TESTS_OBJECTION:
    ok = classer_objection(texte) == attendu
    bons += ok
    print(("ok       " if ok else "À REVOIR  ") + f"{texte!r:44} -> {classer_objection(texte)}")
print(f"\n{bons}/{len(TESTS_OBJECTION)} corrects")

Remarquez l'ordre : « Envoyez-moi un mail avec les **tarifs** » contient un mot de prix, mais c'est bien une demande d'e-mail. **L'ordre des règles est une décision de conception.** Dans la vraie vie, c'est le LLM qui classe, et la règle sert de test de non-régression.

**[Guidé] Exercice 10.4** : écrivez le garde-fou `reponse_conforme(texte)`. Il renvoie `False` si la réponse d'Awa :
- contient un **pourcentage** (`%`) ;
- parle de `remise`, `réduction`, `rabais` ou `gratuit` **sans** « 2 mois offerts » (l'expression exacte « 2 mois offerts » ou « deux mois offerts ») ;
- cite un **montant** autre que 25 000 ou 49 000 (les tarifs).

*Indice* : retirez d'abord les espaces entre les groupes de chiffres (`re.sub(r"(?<=\d)[\s  ](?=\d{3}\b)", "", t)`) pour que « 49 000 » devienne `49000`, puis utilisez `re.findall(r"\d+", t)`. Les petits nombres (2, 3, 15, 30) sont autorisés : ce sont des mois, des utilisateurs, des minutes.

In [ ]:
NOMBRES_AUTORISES = {25000, 49000, 2, 3, 15, 30}

def reponse_conforme(texte):
    t = texte.lower()
    # TODO : refuser « % » ; refuser remise/réduction/rabais/gratuit sans « 2 mois offerts » / « deux mois offerts » ;
    #        refuser tout nombre qui n'est pas dans NOMBRES_AUTORISES (après avoir regroupé « 49 000 » en 49000)
    return True

REPONSES_TEST = [
    ("Je comprends. L'offre Pro est à 49 000 F CFA par mois, et en paiement annuel vous avez 2 mois offerts. Un rendez-vous de 30 minutes permet de voir ce qui convient.", True),
    ("Je peux vous faire 20 % de remise.", False),
    ("Je vous la propose à 30 000 F CFA par mois.", False),
    ("C'est gratuit les trois premiers mois.", False),
    ("Bien sûr, je vous envoie un e-mail. Souhaitez-vous aussi un rendez-vous de 30 minutes ?", True),
    ("Je peux vous accorder une réduction exceptionnelle.", False),
    ("Essentiel est à 25 000 F CFA par mois, jusqu'à 3 utilisateurs.", True),
]
bons = 0
for texte, attendu in REPONSES_TEST:
    ok = reponse_conforme(texte) == attendu
    bons += ok
    print(("ok       " if ok else "À REVOIR  ") + f"{texte[:62]!r:66} -> {reponse_conforme(texte)}")
print(f"\n{bons}/{len(REPONSES_TEST)} corrects")

Ce garde-fou a des **limites** : il ne comprend pas le sens (« je vous l'offre » sans le mot « gratuit » passe), et il ne voit pas les montants écrits en toutes lettres (« trente mille francs »). C'est un **filet de sécurité** simple ; en production, on le complète par un test sur un jeu de conversations (module 12).

### 10.5 Le transfert vers un humain

Awa doit savoir **quand** passer la main, et **comment** :

| Situation | Décision |
|---|---|
| Le prospect **demande** un humain (« passez-moi un conseiller ») | transférer |
| Le prospect est **en colère** (Jean-Pierre) | transférer, sans discuter |
| **Deux échecs** de suite (compréhension, outil) | transférer |
| Le prospect est **hors cible** (Khady, étudiante) | **ne pas** transférer : conclure poliment |
| Prospect **ordinaire** (Mamadou, Aïssatou) | continuer |

Comment : Awa **annonce** le transfert, attend la **fin de sa phrase**, puis transfère avec un **résumé** pour le conseiller, afin que le prospect n'ait **pas à se répéter**.

**[Guidé] Exercice 10.5** : écrivez `doit_transferer(message, nb_echecs)`. Elle renvoie `(True, raison)` ou `(False, "")`. Les raisons possibles : `"demande_humain"`, `"colere"`, `"echecs_repetes"`.

*Indice* : mettez le message en minuscules. Cherchez d'abord les mots de `MOTS_HUMAIN`, puis ceux de `MOTS_COLERE`, puis regardez `nb_echecs >= 2`. Le **premier** critère qui s'applique donne la raison.

In [ ]:
MOTS_HUMAIN = ["un humain", "un conseiller", "une personne", "quelqu'un", "un responsable", "un vrai"]
MOTS_COLERE = ["scandaleux", "inadmissible", "marre", "arnaque", "plainte", "remboursement", "furieux", "mécontent", "nul"]

def doit_transferer(message, nb_echecs):
    t = message.lower()
    # TODO : demande d'un humain -> (True, "demande_humain") ; colère -> (True, "colere") ;
    #        deux échecs ou plus -> (True, "echecs_repetes") ; sinon (False, "")
    return (False, "")

PERSONAS = [
    ("Mamadou", "On est huit salariés, on a une quincaillerie, je voudrais voir ce que ça donne.", 0, False),
    ("Aïssatou", "Nous sommes 120 salariés, je suis la DAF, quel est le tarif Entreprise ?", 0, False),
    ("Jean-Pierre", "C'est inadmissible, j'attends mon remboursement depuis un mois, j'en ai marre !", 0, True),
    ("Khady", "Je suis étudiante, c'est pour un projet de cours, vous avez une version gratuite ?", 0, False),
    ("Prospect", "Je voudrais parler à un conseiller, s'il vous plaît.", 0, True),
    ("Prospect", "Pardon ? Je n'ai pas compris, répétez.", 2, True),
]
bons = 0
for nom, message, echecs, attendu in PERSONAS:
    decision, raison = doit_transferer(message, echecs)
    ok = decision == attendu
    bons += ok
    print(("ok       " if ok else "À REVOIR  ") + f"{nom:12} -> {decision!s:5} {raison}")
print(f"\n{bons}/{len(PERSONAS)} corrects")

Le mot « nul » est **piégeux** : « je n'ai nulle part où noter » le contient, sans aucune colère. Les règles par mots-clés sont fragiles ; en production, c'est le LLM qui juge du ton, et ces règles servent de **filet** et de test.

**Le résumé pour le conseiller.** Un transfert sans contexte oblige le prospect à tout répéter. Awa passe un **résumé court** : qui, quoi, pourquoi, et ce qui a déjà été dit.

In [ ]:
def resume_transfert(prospect, raison, derniers_echanges):
    """Résumé de 3 lignes pour le conseiller humain."""
    lignes = [
        f"Prospect : {prospect.get('nom', 'inconnu')}, {prospect.get('taille_entreprise', '?')} salariés.",
        f"Raison du transfert : {raison}.",
        "Dernier échange : " + " / ".join(derniers_echanges[-2:]),
    ]
    return "\n".join(lignes)

print(resume_transfert({"nom": "Jean-Pierre Diouf", "taille_entreprise": 14}, "colere",
                       ["Awa : Bonjour, ici Awa de Baobab Software.", "Jean-Pierre : C'est inadmissible, j'attends mon remboursement."]))
print("\nMessage avant le transfert :")
print("  « Je comprends, je vous passe un conseiller tout de suite. Je lui transmets votre dossier, vous n'aurez pas à répéter. »")

**Côté LiveKit.** Pour un appel **téléphonique** (SIP), le transfert passe par l'API serveur : `lk.sip.transfer_sip_participant(api.TransferSIPParticipantRequest(participant_identity=..., room_name=..., transfer_to="tel:+221..."))`. La requête contient aussi `play_dialtone`, `headers` et `ringing_timeout` (noms vérifiés dans livekit-api 1.2.2). Cela suppose un appel entrant SIP ; sur un appel web, il n'y a rien à transférer par téléphone : on propose un **rappel**. Je n'ai pas vérifié le détail d'un transfert « accompagné » (le conseiller est briefé avant de prendre l'appel) : consultez la documentation LiveKit du jour.

### 10.6 Accents et choix du STT

Un STT qui marche bien **en moyenne** peut mal marcher pour **vos** locuteurs : accent, débit, bruit de ligne, mots wolof au milieu d'une phrase, noms propres locaux. Et les erreurs ne se valent pas : se tromper sur « euh » est sans conséquence, se tromper sur **un chiffre du numéro de téléphone** ou sur **une lettre de l'e-mail** fait perdre le prospect (module 5).

La méthode : faire lire les **mêmes phrases** par plusieurs locuteurs de l'équipe, faire transcrire chaque enregistrement par **2 ou 3 STT**, puis comparer le **WER** (module 5) et le **taux d'entités perdues**, **par locuteur**.

Dans ce notebook, les transcriptions sont **simulées** : chaque STT a un **profil d'erreur fictif** par locuteur. Ce n'est pas un benchmark. Aucun chiffre ici ne dit qu'un vrai STT est meilleur qu'un autre.

In [ ]:
PHRASES = [
    {"ref": "je m'appelle fatou sow et mon numéro est le 77 616 06 16", "entites": ["fatou sow", "776160616"]},
    {"ref": "nous sommes huit salariés et notre budget est de 49 000 francs par mois", "entites": ["huit", "49000"]},
    {"ref": "je voudrais un rendez-vous mardi à dix heures trente", "entites": ["mardi", "dix heures trente"]},
    {"ref": "mon e-mail est fatou point sow arobase baobab point sn", "entites": ["fatou point sow", "arobase baobab point sn"]},
]
LOCUTEURS = ["Locuteur 1", "Locuteur 2", "Locuteur 3"]
STT_NOMS = ["STT A", "STT B", "STT C"]

# profil d'erreur FICTIF : probabilité de se tromper sur un mot, par STT et par locuteur
PROFILS_ERREUR = {
    "STT A": {"Locuteur 1": 0.04, "Locuteur 2": 0.14, "Locuteur 3": 0.09},
    "STT B": {"Locuteur 1": 0.07, "Locuteur 2": 0.08, "Locuteur 3": 0.07},
    "STT C": {"Locuteur 1": 0.10, "Locuteur 2": 0.10, "Locuteur 3": 0.16},
}
CONFUSIONS = {"fatou": "fatoue", "sow": "sot", "huit": "huis", "trente": "trant", "dix": "six", "mardi": "mercredi",
              "point": "pont", "arobase": "à rôle base", "salariés": "salarié", "francs": "franc", "budget": "budjet",
              "numéro": "numéros", "rendez-vous": "rendez vous"}

def degrader(texte, p, rng):
    """Simule une transcription : chaque mot est remplacé, supprimé ou gardé ; chaque chiffre peut changer."""
    sortie = []
    for mot in texte.split():
        r = rng.random()
        if r < p / 3:
            continue                                    # mot supprimé
        if r < p:
            if mot.isdigit() and rng.random() < 0.7:
                mot = str((int(mot) + rng.choice([1, 2, 3, 7])) % (10 ** len(mot))).zfill(len(mot))
            else:
                mot = CONFUSIONS.get(mot, mot + "s")
        sortie.append(mot)
    return " ".join(sortie)

def fabriquer_corpus(repetitions=10, graine=7):
    rng = random.Random(graine)
    corpus = []
    for stt in STT_NOMS:
        for loc in LOCUTEURS:
            for ph in PHRASES:
                for _ in range(repetitions):
                    corpus.append({"stt": stt, "locuteur": loc, "ref": ph["ref"], "entites": ph["entites"],
                                   "hyp": degrader(ph["ref"], PROFILS_ERREUR[stt][loc], rng)})
    return corpus

CORPUS = fabriquer_corpus()
print(len(CORPUS), "transcriptions simulées. Exemple :")
ex = CORPUS[3]
print("  ref :", ex["ref"]); print("  hyp :", ex["hyp"], f"({ex['stt']}, {ex['locuteur']})")

**Reprise du module 5 : le WER.** Distance d'édition entre les mots de la référence et ceux de l'hypothèse (substitutions, suppressions, insertions), divisée par le nombre de mots de la référence.

In [ ]:
def normaliser(texte):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s'’-]", " ", texte.lower())).strip()

def wer(ref, hyp):
    r, h = normaliser(ref).split(), normaliser(hyp).split()
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prec, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cout = 0 if r[i - 1] == h[j - 1] else 1
            prec, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prec + cout)
    return d[len(h)] / max(len(r), 1)

print("WER de test :", round(wer("le rendez-vous est mardi", "le rendez vous est mercredi"), 2), "(attendu 0.75 : le mot rendez-vous devient 2 mots)")

**[Guidé] Exercice 10.6** : écrivez `entites_perdues(entites, hyp)`, qui renvoie **combien d'entités** de la liste ne se retrouvent **pas** dans la transcription.

*Indice* : comparez sans les espaces, tirets et points. Une fonction `compacter(s)` qui fait `re.sub(r"[\s.\-]", "", s.lower())` règle le cas « 77 616 06 16 » contre `776160616`. Une entité est **perdue** si `compacter(entite)` n'est pas dans `compacter(hyp)`.

In [ ]:
def compacter(s):
    return re.sub(r"[\s.\-]", "", s.lower())

def entites_perdues(entites, hyp):
    # TODO : compter les entités dont compacter(entite) n'est pas dans compacter(hyp)
    return 0

TESTS_ENTITES = [
    (entites_perdues(["776160616"], "mon numéro est le 77 616 06 16"), 0, "numéro bien transcrit"),
    (entites_perdues(["776160616"], "mon numéro est le 77 616 06 17"), 1, "un chiffre faux"),
    (entites_perdues(["fatou sow", "huit"], "fatou sot et huit"), 1, "un nom faux"),
]
for obtenu, attendu, nom in TESTS_ENTITES:
    print(("ok       " if obtenu == attendu else "À REVOIR  ") + f"{nom} : {obtenu} perdue(s), attendu {attendu}")

In [ ]:
def tableau(corpus):
    """WER moyen et taux d'entités perdues, par STT et par locuteur."""
    res = {}
    for stt in STT_NOMS:
        for loc in LOCUTEURS:
            lignes = [c for c in corpus if c["stt"] == stt and c["locuteur"] == loc]
            w = float(np.mean([wer(c["ref"], c["hyp"]) for c in lignes]))
            total = sum(len(c["entites"]) for c in lignes)
            perdues = sum(entites_perdues(c["entites"], c["hyp"]) for c in lignes)
            res[(stt, loc)] = {"wer": w, "entites": perdues / total}
    return res

T = tableau(CORPUS)
print(f"{'':8}" + "".join(f"{loc:>22}" for loc in LOCUTEURS))
for stt in STT_NOMS:
    print(f"{stt:8}" + "".join(f"   WER {T[(stt, l)]['wer']*100:4.1f} % · ent. {T[(stt, l)]['entites']*100:3.0f} %" for l in LOCUTEURS))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
largeur = 0.25
for k, (cle, titre) in enumerate((("wer", "WER moyen (%)"), ("entites", "Entités perdues (%)"))):
    for i, stt in enumerate(STT_NOMS):
        axes[k].bar(np.arange(3) + (i - 1) * largeur, [T[(stt, l)][cle] * 100 for l in LOCUTEURS], largeur, label=stt, color=[NAVY, TEAL, AMBER][i])
    axes[k].set_xticks(range(3)); axes[k].set_xticklabels(LOCUTEURS); axes[k].set_title(titre)
axes[0].legend(frameon=False)
fig.suptitle("Profils d'erreur FICTIFS : à refaire sur vos enregistrements", fontsize=9, color="#5E6779")
plt.tight_layout(); plt.show()

**Lire le tableau.** Il montre trois choses que le WER moyen cache :
1. **Le meilleur STT dépend du locuteur** : ici, aucun STT n'est le meilleur pour les trois locuteurs.
2. **La moyenne ment** : un STT peut avoir un bon WER moyen et mal servir l'un des locuteurs.
3. **Les entités souffrent plus que le WER ne le laisse penser** : avec un WER d'environ 10 %, on perd ici de 18 à 22 % des entités, car un seul chiffre faux ruine un numéro entier.

**Les langues locales.** Un STT peut ne pas supporter le **wolof** ou un autre parler local, et accepter en revanche des **mots wolof** dans une phrase française (noms de quartiers, de personnes). Vérifiez la **liste des langues supportées** de chaque fournisseur le jour J : elle change, et je ne l'ai pas vérifiée pour ce module.

**Faire la comparaison pour de vrai (optionnel).** Protocole pour l'atelier :
1. Chacun lit les **4 phrases** ci-dessus, à son rythme naturel, dans un endroit **calme** puis dans un endroit **bruyant**.
2. Enregistrez avec le **même appareil** pour tous (téléphone, par exemple), sans retouche.
3. Obtenez le **consentement** des participants pour l'enregistrement et supprimez les fichiers après l'atelier (module 13).
4. Faites transcrire chaque fichier par 2 ou 3 STT, puis réutilisez `wer` et `entites_perdues`.

Le code ci-dessous attend vos **fonctions de transcription** (une par STT). Sans elles, il ne fait rien.

In [ ]:
def comparer_enregistrements(dossier, transcripteurs, phrases=PHRASES):
    """dossier : fichiers nommés locuteur_phrase.wav (par exemple fatou_0.wav).
    transcripteurs : {"STT A": fonction(chemin_audio) -> texte, ...}. À brancher sur vos API."""
    import os
    if not os.path.isdir(dossier) or not transcripteurs:
        print("Aucun dossier d'enregistrements ou aucun transcripteur : étape optionnelle ignorée.")
        return None
    corpus = []
    for nom in sorted(os.listdir(dossier)):
        if nom.endswith(".wav"):
            loc, idx = nom[:-4].rsplit("_", 1)
            ph = phrases[int(idx)]
            for stt, transcrire in transcripteurs.items():
                corpus.append({"stt": stt, "locuteur": loc, "ref": ph["ref"], "entites": ph["entites"],
                               "hyp": transcrire(os.path.join(dossier, nom))})
    return corpus

comparer_enregistrements("enregistrements_equipe", {})

### 10.7 Atelier : une Awa fluide

Assemblez les briques de ce module sur quatre situations. Le code ci-dessous **rejoue** une conversation de chaque type avec vos fonctions ; lisez les sorties.

In [ ]:
def jouer_situation(titre, tour_prospect, duree_outil=None, echecs=0):
    print(f"--- {titre} ---")
    print("Prospect :", tour_prospect)
    decision, raison = doit_transferer(tour_prospect, echecs)
    if decision:
        print(f"Awa      : Je comprends, je vous passe un conseiller. (transfert : {raison})")
        return
    objection = classer_objection(tour_prospect)
    if objection != "aucune":
        print(f"Awa      : [objection détectée : {objection}]")
    if duree_outil is not None:
        d = []
        evts = evenements_attente(duree_outil, SEUILS)
        for t, nom in evts:
            if nom in ("filler", "relance"):
                print(f"Awa      : ({t:.1f} s) {choisir_filler('agenda', d, 0 if nom == 'filler' else 1)}")
            elif nom == "timeout":
                print(f"Awa      : ({t:.1f} s) Je n'arrive pas à joindre l'agenda. Je peux vous rappeler, ou vous passer un conseiller.")
            else:
                print(f"Awa      : ({t:.1f} s) J'ai un créneau mardi à dix heures. Cela vous convient ?")
    print()

jouer_situation("Agenda lent de 5 s", "Je voudrais un rendez-vous mardi.", duree_outil=5.0)
jouer_situation("Objection de prix", "C'est trop cher pour une quincaillerie de huit salariés.")
jouer_situation("Demande d'un e-mail", "Envoyez-moi un e-mail, je regarderai.")
jouer_situation("Client mécontent", "C'est inadmissible, j'en ai marre !")
jouer_situation("Agenda en panne", "Je voudrais un rendez-vous jeudi.", duree_outil=12.0)

**Questions pour l'atelier** (à répondre dans une cellule de texte) :
1. Avec l'agenda à **5 s**, quels seuils choisissez-vous (filler, relance, timeout) et quel est le **plus long silence** perçu ? Comparez avec le cas sans filler.
2. Écrivez la **réponse d'Awa** à « c'est cher » et à « envoyez-moi un e-mail », et vérifiez-les avec `reponse_conforme`.
3. Pour **Jean-Pierre**, quel **message** Awa dit-elle avant le transfert, et quel **résumé** reçoit le conseiller ?
4. Quel **STT** choisissez-vous pour votre équipe, **pourquoi**, et avec quels **deux chiffres** (WER et entités perdues, pour le locuteur le plus mal servi) ?

**Livrable du jour** : une Awa fluide malgré l'attente et les interruptions (`awa_agent.py` avec `AWA_PATIENCE`) et le **tableau comparatif des STT** de votre équipe.

### À retenir

- Un silence de quelques secondes fait croire à une **ligne coupée** : on le **découpe** avec un préambule, un **filler**, une **relance** et un **timeout** (seuils : repères indicatifs, à régler sur vos appels).
- Le **timeout** doit déboucher sur une **solution de repli** (rappel, conseiller), pas sur un silence.
- Le filler est **court**, **adapté à l'outil** et **jamais répété** de façon rapprochée. Il reste **interruptible**.
- Sur une objection de prix, Awa **ne négocie pas** : tarif exact, seule réduction (2 mois offerts en annuel), rendez-vous. Un **garde-fou** sur la sortie empêche d'inventer une remise.
- Elle **transfère** sur demande d'un humain, colère ou échecs répétés, avec un **résumé** pour que le prospect ne se répète pas.
- Pour choisir un **STT**, on compare **WER** et **entités perdues**, **par locuteur**, sur **vos** enregistrements.

---
### Quiz du Module 10

**Q1.** Pourquoi Awa dit-elle une courte phrase avant d'appeler un outil lent ?
a) pour que le LLM ait plus de temps de calcul · b) parce qu'un silence de quelques secondes fait croire à l'appelant que la ligne est coupée · c) pour réduire le coût de l'outil · d) parce que l'outil répond alors plus vite

**Q2.** L'agenda met 5 secondes à répondre. Quel comportement est le plus fluide ?
a) rester silencieuse jusqu'au résultat · b) répéter le même filler toutes les secondes · c) raccrocher et rappeler · d) un court filler après environ une seconde, une relance plus tard, et un timeout avec une solution de repli

**Q3.** Le prospect dit « c'est cher ». Quelle réponse est conforme à la politique d'Awa ?
a) reconnaître l'objection, redonner le tarif exact, rappeler les deux mois offerts en paiement annuel, proposer un rendez-vous · b) offrir 20 % de remise pour le garder · c) transférer immédiatement vers un humain · d) ignorer l'objection et poser la question suivante

**Q4.** Dans quels cas Awa transfère-t-elle vers un humain ?
a) dès que le prospect demande un prix · b) jamais, elle doit toujours conclure seule · c) sur demande explicite d'un humain, en cas de colère, ou après des échecs répétés · d) toutes les cinq minutes, par précaution

**Q5.** Comment comparer des STT sur les accents de l'équipe ?
a) un seul locuteur suffit · b) lire le WER global d'un benchmark en anglais · c) choisir le moins cher · d) faire lire les mêmes phrases par plusieurs locuteurs, et comparer le WER et les entités perdues par locuteur

In [ ]:
mes_reponses_m10 = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses

In [ ]:
def corriger_quiz_module10(reponses):
    attendu = {1: "b", 2: "d", 3: "a", 4: "c", 5: "d"}
    explications = {
        1: "Un silence de quelques secondes fait croire à une ligne coupée. Le préambule et le filler le comblent sans passer par le LLM.",
        2: "On découpe l'attente : filler vers 1 s, relance vers 4 s, timeout vers 8 s avec une solution de repli (repères indicatifs).",
        3: "Awa n'invente jamais de remise : tarif exact, seule réduction (2 mois offerts en annuel), puis un rendez-vous.",
        4: "Demande d'un humain, colère ou échecs répétés. Le prix seul n'est pas une raison de transférer.",
        5: "Le meilleur STT dépend du locuteur, et les erreurs sur les chiffres et les noms coûtent plus que les autres : on compare WER et entités, par locuteur.",
    }
    score = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score}/5")

corriger_quiz_module10(mes_reponses_m10)